# Notebook 05: Speed comparisons on one GPU (the other GPU runs Notebooks 01 and 03)

**Scenario:** the two GPUs work independently. This notebook is pinned to **GPU 0** and holds a quantized DiffusionGemma for timing; Notebooks 01 and 03 run at the same time on **GPU 1** (set `GPU_ID = 1` in their first cell).

Speed comparisons belong on a single GPU: a model split across two GPUs adds cross-GPU transfers that blur the timing.

**What this notebook measures**

| Part | Measurement | Why |
|---|---|---|
| A | Sampling-step cost on **real** logits vs. batch size, split into the two places it happens: the scheduler (entropy, commits, sampling) and self-conditioning (probabilities x embeddings) | The core speed and memory claim (H1), on real data instead of synthetic |
| B | End-to-end generation time, steps and peak memory vs. batch size, stock vs. certified scheduler | The baseline before the self-conditioning and stopping patches; rerun after patching |

**Precision.** 4-bit by default (about 15 GB); `int8` (about 26 GB) is closer to full precision but leaves less room for large batches. Keep the same precision for every variant in a comparison.

## 0. Pin this notebook to one GPU (run first, before anything imports torch)

In [ ]:
import os
GPU_ID = 0
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)
print('this notebook sees only physical GPU', GPU_ID, '(it appears as cuda:0 inside the notebook)')

## 1. Install and shared code

In [ ]:
!pip install -q -U "transformers>=5.12" diffusers accelerate bitsandbytes pandas

In [ ]:
%%writefile dg_sampling.py
"""Shared sampler code for the DiffusionGemma top-k / certified sampling project.

Everything here is pure PyTorch and model-agnostic except TopKEntropyBoundScheduler,
which subclasses the diffusers EntropyBoundScheduler.
"""
import math
import torch
import torch.nn.functional as F

BUDGET, E_STOP = 0.1, 0.005


# ----------------------------------------------------------------------------- basic pieces
def chunked_logsumexp(x, tau, chunk=32_768):
    """log-sum-exp of x / tau over the last dim in fp32, one vocabulary chunk at a time."""
    out = None
    for s in range(0, x.shape[-1], chunk):
        part = torch.logsumexp(x[..., s:s + chunk].float() / tau, dim=-1)
        out = part if out is None else torch.logaddexp(out, part)
    return out


def exact_entropy(rows, tau):
    lp = torch.log_softmax(rows.float() / tau, dim=-1)
    return -(lp.exp() * lp).sum(-1)


def n_commit(e, b=BUDGET):
    s, _ = torch.sort(e)
    return int(((s.cumsum(0) - s) <= b).sum())


def commit_exact(e, b=BUDGET):
    order = torch.argsort(e, stable=True)
    keep = torch.zeros_like(e, dtype=torch.bool)
    keep[order[:n_commit(e, b)]] = True
    return keep


def topk_stats(logits, tau, k):
    """Exact top-k probabilities, tail mass m and a guaranteed entropy interval [lo, hi]."""
    Vv = logits.shape[-1]
    vals, idx = torch.topk(logits, k, dim=-1)
    lse = chunked_logsumexp(logits, tau)
    lp = vals.float() / tau - lse.unsqueeze(-1)
    p = lp.exp()
    h = -(p * lp).sum(-1)
    m = (1.0 - p.sum(-1)).clamp(0.0, 1.0)
    nmlm = -m * torch.log(m.clamp_min(1e-30))
    hi = h + nmlm + (m * math.log(Vv - k) if Vv > k else 0.0)
    lo = h - m * torch.log(torch.minimum(m, p[..., -1]).clamp_min(1e-30))
    return dict(idx=idx, p=p, m=m, lse=lse, h=h, lo=lo, hi=hi)


def topk_mix(st, E):
    """Self-conditioning mix over the top-k only: sum_k p_k * E[idx_k]."""
    B, Cc, k = st['p'].shape
    w = st['p'].reshape(-1, k).to(E.dtype).contiguous()
    try:
        out = F.embedding_bag(st['idx'].reshape(-1, k), E, per_sample_weights=w, mode='sum')
    except RuntimeError:   # some builds lack embedding_bag for this dtype: gather + weighted sum instead
        out = torch.einsum('nk,nkd->nd', w, E[st['idx'].reshape(-1, k)])
    return out.view(B, Cc, -1)


# ----------------------------------------------------------------------------- certified decisions
def certified_commit_row(lo, hi, refine, b=BUDGET, chunk=8):
    """Returns (commit mask identical to commit_exact(true entropies), refined slot count).
    Slots whose intervals are too loose to decide are refined with refine(pos) -> exact entropies."""
    lo, hi = lo.clone(), hi.clone()
    exact = torch.zeros_like(lo, dtype=torch.bool)
    refined = 0
    while True:
        if bool(exact.all()):
            return commit_exact(hi, b), refined
        n_lo, n_hi = n_commit(lo, b), n_commit(hi, b)
        if n_lo == n_hi:
            order = torch.argsort(hi, stable=True)
            S, R = order[:n_lo], order[n_lo:]
            if R.numel() == 0 or hi[S].max() < lo[R].min():
                keep = torch.zeros_like(exact)
                keep[S] = True
                return keep, refined
            amb = torch.cat([S[hi[S] >= lo[R].min()], R[lo[R] <= hi[S].max()]])
        else:
            amb = torch.argsort(lo, stable=True)[:n_lo + 1]
        amb = amb[~exact[amb]]
        if amb.numel() == 0:
            amb = (~exact).nonzero().squeeze(-1)
        if amb.numel() > chunk:
            amb = amb[torch.argsort(hi[amb] - lo[amb], descending=True)[:chunk]]
        e = refine(amb)
        lo[amb], hi[amb] = e, e
        exact[amb] = True
        refined += amb.numel()


def certified_stop_row(lo, hi, refine, e_stop=E_STOP, chunk=32):
    lo, hi = lo.clone(), hi.clone()
    exact = torch.zeros_like(lo, dtype=torch.bool)
    refined = 0
    while True:
        if hi.mean() <= e_stop:
            return True, refined
        if lo.mean() > e_stop:
            return False, refined
        cand = (~exact).nonzero().squeeze(-1)
        if cand.numel() == 0:
            return bool(hi.mean() <= e_stop), refined
        pick = cand[torch.argsort(hi[cand] - lo[cand], descending=True)[:chunk]]
        e = refine(pick)
        lo[pick], hi[pick] = e, e
        exact[pick] = True
        refined += pick.numel()


def certify_metrics(L, tau, ks=(16, 64, 256, 1024), b=BUDGET, e_stop=E_STOP):
    """For one step's logits L [B, C, V]: decision mismatches (must be 0) and refinement fractions per k."""
    e_true = exact_entropy(L, tau)
    rows = []
    for k in ks:
        st = topk_stats(L, tau, k)
        out = dict(k=k, commit_mismatch=0, stop_mismatch=0, refined_commit=0, refined_stop=0,
                   tail_mass_mean=st['m'].mean().item())
        for bi in range(L.shape[0]):
            refine = lambda pos, bi=bi: exact_entropy(L[bi, pos], tau)
            keep, r1 = certified_commit_row(st['lo'][bi], st['hi'][bi], refine, b)
            stop, r2 = certified_stop_row(st['lo'][bi], st['hi'][bi], refine, e_stop)
            out['commit_mismatch'] += int((keep != commit_exact(e_true[bi], b)).any())
            out['stop_mismatch'] += int(stop != bool(e_true[bi].mean() <= e_stop))
            out['refined_commit'] += r1
            out['refined_stop'] += r2
        n = L.shape[0] * L.shape[1]
        out['refined_commit_frac'] = out.pop('refined_commit') / n
        out['refined_stop_frac'] = out.pop('refined_stop') / n
        rows.append(out)
    return rows


# ----------------------------------------------------------------------------- diffusers integration
def calibrate_tau(sched, n_steps, device='cuda'):
    """Recover the stock scheduler's exact temperature at every timestep from pred_logits = logits / tau."""
    try:
        sched.set_timesteps(n_steps)
        ts = getattr(sched, 'timesteps', None)
        timesteps = [int(t) for t in ts] if ts is not None and len(ts) == n_steps else list(range(n_steps))
    except Exception:
        timesteps = list(range(n_steps))
    x = torch.randn(1, 8, 1024, device=device) * 3 + 1.0
    sample = torch.randint(1024, (1, 8), device=device)
    taus = [round((x / sched.step(x, t, sample).pred_logits.float()).median().item(), 6) for t in timesteps]
    return timesteps, taus


def make_scheduler_class():
    from diffusers import EntropyBoundScheduler
    try:
        from diffusers.schedulers.scheduling_entropy_bound import EntropyBoundSchedulerOutput
    except ImportError:
        from diffusers import EntropyBoundSchedulerOutput

    class TopKEntropyBoundScheduler(EntropyBoundScheduler):
        """Drop-in scheduler. mode: 'parent' | 'full' | 'topk' | 'certified'.
        Samples come from the top-k (no tail branch) in 'topk' and 'certified' modes."""

        def __init__(self, *args, mode='certified', k=64, tail='hi', pred_logits_mode='full',
                     tau_table=None, timesteps=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.mode, self.k, self.tail, self.pred_logits_mode = mode, k, tail, pred_logits_mode
            self.tau_by_t = dict(zip(timesteps, tau_table)) if (tau_table and timesteps) else None
            self.stats = []

        def _tau(self, timestep):
            t = int(timestep)
            if self.tau_by_t and t in self.tau_by_t:
                return self.tau_by_t[t]
            n = self.config.num_inference_steps
            return self.config.t_max - (self.config.t_max - self.config.t_min) * min(t, n - 1) / max(n - 1, 1)

        def step(self, model_output, timestep, sample, entropy_bound=None, generator=None, return_dict=True):
            if self.mode == 'parent':
                return super().step(model_output, timestep, sample, entropy_bound=entropy_bound,
                                    generator=generator, return_dict=return_dict)
            logits = model_output
            b = self.config.entropy_bound if entropy_bound is None else entropy_bound
            tau = self._tau(timestep)
            B, Cc, Vv = logits.shape
            g = generator if (generator is not None and generator.device == logits.device) else None
            refined = 0
            if self.mode == 'full':
                lp = torch.log_softmax(logits.float() / tau, -1)
                p = lp.exp()
                ent = -(p * lp).sum(-1)
                keep = torch.stack([commit_exact(ent[i], b) for i in range(B)])
                tok = torch.multinomial(p.view(-1, Vv), 1, generator=g).view(B, Cc)
                lse = torch.logsumexp(logits.float() / tau, -1)
            else:
                st = topk_stats(logits, tau, self.k)
                lse = st['lse']
                if self.mode == 'topk':
                    if self.tail == 'renorm':
                        q = st['p'] / st['p'].sum(-1, keepdim=True)
                        ent = -(q * torch.log(q.clamp_min(1e-30))).sum(-1)
                    else:
                        ent = {'lo': st['lo'], 'hi': st['hi'], 'mid': 0.5 * (st['lo'] + st['hi']), 'none': st['h']}[self.tail]
                    keep = torch.stack([commit_exact(ent[i], b) for i in range(B)])
                elif self.mode == 'certified':
                    rows = []
                    for i in range(B):
                        refine = lambda pos, i=i: exact_entropy(logits[i, pos], tau)
                        kp, r = certified_commit_row(st['lo'][i], st['hi'][i], refine, b)
                        rows.append(kp)
                        refined += r
                    keep = torch.stack(rows)
                else:
                    raise ValueError(self.mode)
                pick = torch.multinomial(st['p'].reshape(-1, self.k), 1, generator=g).view(B, Cc)
                tok = st['idx'].gather(-1, pick.unsqueeze(-1)).squeeze(-1)
            noise = torch.randint(Vv, (B, Cc), device=logits.device, generator=g)
            prev = torch.where(keep, tok, noise)
            sampled_probs = torch.exp(logits.gather(-1, tok.unsqueeze(-1)).squeeze(-1).float() / tau - lse)
            if self.pred_logits_mode == 'full':
                pred_logits = logits.float() / tau
            else:
                pred_logits = torch.full(logits.shape, float('-inf'), device=logits.device, dtype=torch.float32)
                vals, idx = torch.topk(logits, self.k, -1)
                pred_logits.scatter_(-1, idx, vals.float() / tau)
            self.stats.append(dict(t=int(timestep), committed=keep.sum(-1).float().mean().item(), refined=refined))
            out = EntropyBoundSchedulerOutput(prev_sample=prev, accepted_index=keep, sampled_tokens=tok,
                                              sampled_probs=sampled_probs, pred_logits=pred_logits)
            return out if return_dict else out.to_tuple()

    return EntropyBoundScheduler, TopKEntropyBoundScheduler


class StepRecorder:
    """diffusers callback: counts denoising steps and optionally runs a function on each step's logits."""
    def __init__(self, on_logits=None):
        self.on_logits = on_logits
        self.reset()

    def reset(self):
        self.steps = 0

    def __call__(self, pipe, step, timestep, kw):
        self.steps += 1
        if self.on_logits is not None and 'logits' in kw:
            self.on_logits(step, timestep, kw['logits'])
        return {'canvas': kw['canvas']}


# ----------------------------------------------------------------------------- small eval helpers
def last_number(s):
    import re
    nums = re.findall(r'-?\d[\d,]*\.?\d*', s)
    return nums[-1].replace(',', '').rstrip('.') if nums else None


def gsm8k_gold(ans):
    return ans.split('####')[-1].strip().replace(',', '')

In [ ]:
import time, gc, json
import torch, pandas as pd, matplotlib.pyplot as plt, transformers, diffusers
import torch.nn.functional as F
import dg_sampling as dgs
print(torch.cuda.get_device_name(0), '| transformers', transformers.__version__, '| diffusers', diffusers.__version__)
MODEL_ID = 'google/diffusiongemma-26B-A4B-it'
N_STEPS = 48
PRECISION = '4bit'          # or 'int8'
OUT_DIR = './runs/nb05'
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load the quantized model on this GPU

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, DiffusionGemmaForBlockDiffusion
from diffusers import DiffusionGemmaPipeline

q = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16,
                        bnb_4bit_use_double_quant=True)
     if PRECISION == '4bit' else BitsAndBytesConfig(load_in_8bit=True))
t0 = time.time()
model = DiffusionGemmaForBlockDiffusion.from_pretrained(MODEL_ID, quantization_config=q, dtype=torch.bfloat16,
                                                        device_map={'': 0}, low_cpu_mem_usage=True)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f'loaded in {time.time() - t0:.0f}s | {torch.cuda.memory_allocated() / 2**30:.1f} GB allocated')

StockSched, TopKSched = dgs.make_scheduler_class()
stock = StockSched(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS)
TIMESTEPS, TAU_TABLE = dgs.calibrate_tau(stock, N_STEPS)
pipe = DiffusionGemmaPipeline(model=model, scheduler=stock, processor=processor)

E = model.get_input_embeddings().weight.detach()       # [V, d] embedding table used by self-conditioning
print('embedding table:', tuple(E.shape), E.dtype)

## 3. Part A: sampling-step cost on real logits

Grab real logits from three points of one generation (early, middle, late step), then tile them to batch B and time each piece of the sampler in isolation:

- **scheduler, stock**: diffusers' `EntropyBoundScheduler.step`
- **scheduler, certified k64**: the drop-in replacement
- **self-conditioning, full**: softmax probabilities x the full embedding table (what the model does today), in bf16 (the optimized form, on tensor cores)
- **self-conditioning, top-k64**: the weighted sum over 64 embedding rows

Out-of-memory results are recorded as missing; where the full version stops fitting is itself a result.

In [ ]:
grab = {}
def keep_some(step, timestep, L):
    if step in (0, 4, 10) and step not in grab:
        grab[step] = (int(timestep), L.detach().clone())
rec = dgs.StepRecorder(on_logits=keep_some)
pipe(prompt='Explain why the sky is blue and why sunsets are red, in one paragraph.', gen_length=256,
     num_inference_steps=N_STEPS, generator=torch.Generator('cuda').manual_seed(0),
     callback_on_step_end=rec, callback_on_step_end_tensor_inputs=['canvas', 'logits'])
print('captured steps:', {k: tuple(v[1].shape) for k, v in grab.items()}, '| total steps', rec.steps)

In [ ]:
def timed(fn, reps=10, warmup=3):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    ts = []
    for _ in range(reps):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    return float(torch.tensor(ts).median()), (torch.cuda.max_memory_allocated() - base) / 2**30

cert = TopKSched(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS,
                 mode='certified', k=64, tau_table=TAU_TABLE, timesteps=TIMESTEPS)
tau_of = dict(zip(TIMESTEPS, TAU_TABLE))

def mix_full(L, tau):
    p = torch.softmax(L.float() / tau, -1).to(E.dtype)
    return p @ E

def mix_topk(L, tau, k=64):
    return dgs.topk_mix(dgs.topk_stats(L, tau, k), E)

OOM = (torch.cuda.OutOfMemoryError, RuntimeError)
rows = []
for step, (t, L1) in grab.items():
    tau = tau_of.get(t, TAU_TABLE[min(step, len(TAU_TABLE) - 1)])
    for B in [1, 2, 4, 8, 16, 32]:
        try:
            L = L1.expand(B, -1, -1).contiguous()
        except OOM:
            break
        sample = torch.zeros(L.shape[:2], dtype=torch.long, device=L.device)
        parts = {
            'scheduler: stock':        lambda: stock.step(L, t, sample),
            'scheduler: certified k64': lambda: cert.step(L, t, sample),
            'self-cond: full (bf16)':   lambda: mix_full(L, tau),
            'self-cond: top-k64':       lambda: mix_topk(L, tau),
        }
        for name, fn in parts.items():
            try:
                ms, gb = timed(fn)
            except OOM:
                ms, gb = float('nan'), float('nan')
            rows.append(dict(step=step, batch=B, part=name, ms=ms, peak_extra_GB=gb))
            torch.cuda.empty_cache()
        del L, sample; gc.collect(); torch.cuda.empty_cache()

partA = pd.DataFrame(rows)
partA.to_csv(f'{OUT_DIR}/sampler_parts_real_logits.csv', index=False)
display(partA.pivot_table(index='batch', columns='part', values='ms', aggfunc='mean').round(2))
display(partA.pivot_table(index='batch', columns='part', values='peak_extra_GB', aggfunc='mean').round(2))

In [ ]:
tot = partA.copy()
tot['variant'] = tot.part.map({'scheduler: stock': 'full sampler', 'self-cond: full (bf16)': 'full sampler',
                               'scheduler: certified k64': 'certified top-k', 'self-cond: top-k64': 'certified top-k'})
tot = tot.groupby(['batch', 'variant']).ms.mean().unstack()
fig, ax = plt.subplots(figsize=(6, 4))
for c in tot.columns:
    ax.plot(tot.index, tot[c], marker='o', label=c)
ax.set(xscale='log', yscale='log', xlabel='batch size', ylabel='ms per step (scheduler + self-conditioning)',
       title='Sampling cost per denoising step, real logits')
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/sampler_cost.png', dpi=150); plt.show()

Note on the stock scheduler: it is diffusers' straightforward implementation. The fair comparison for the paper also includes the best optimized full-vocabulary version (compiled, fused where possible); Notebook 01's `full (optimized)` variant covers that on synthetic logits.

## 4. Part B: end-to-end generation vs. batch size

Stock vs. certified scheduler inside the full pipeline. Until the self-conditioning and stopping patches land, both variants still do full-vocabulary self-conditioning inside the model, so **expect only a small end-to-end difference here**: this run is the baseline to beat after patching. Steps per canvas must match between the two.

In [ ]:
PROMPT = 'Explain how a rainbow forms, step by step.'
plain = dgs.StepRecorder()
rowsB = []
for name, sch in [('stock', stock), ('certified k64', cert)]:
    pipe.scheduler = sch
    for B in [1, 2, 4, 8]:
        prompts = [PROMPT] * B
        try:
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); plain.reset()
            t0 = time.time()
            out = pipe(prompt=prompts, gen_length=256, num_inference_steps=N_STEPS,
                       generator=torch.Generator('cuda').manual_seed(0),
                       callback_on_step_end=plain, callback_on_step_end_tensor_inputs=['canvas'])
            torch.cuda.synchronize(); dt = time.time() - t0
            rowsB.append(dict(variant=name, batch=B, seconds=dt, steps=plain.steps, texts=len(out.texts),
                              peak_GB=torch.cuda.max_memory_allocated() / 2**30))
        except Exception as ex:
            rowsB.append(dict(variant=name, batch=B, seconds=float('nan'), steps=None, texts=None,
                              peak_GB=float('nan'), error=f'{type(ex).__name__}: {str(ex)[:80]}'))
        torch.cuda.empty_cache()
pipe.scheduler = stock
partB = pd.DataFrame(rowsB)
partB.to_csv(f'{OUT_DIR}/end_to_end_batch.csv', index=False)
display(partB)

If batched prompts raise an error, the pipeline may expect a different input format for batches; check `inspect.signature(DiffusionGemmaPipeline.__call__)` and the diffusers docs, then adjust `prompt=`.